# Test filter_service and image_filters

In [1]:
import sys
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: C:\Users\z004ps0b\Documents\dermascan


In [2]:
from preprocessing import image_filters
from preprocessing.filter_service import FilterService
from dataset_manager import DatasetManager 

In [3]:
filter_service = FilterService(
    data_dir=str(proj_dir / "data")
)

dataset_manager = DatasetManager(
    data_dir=str(proj_dir / "data")
)

### Test applying and saving filters

#### Apply all filters for single image

In [4]:
res = filter_service.apply_texture_filters(image_id="ISIC_0024306", filter_list=[image_filters.glcm_filter, image_filters.gabor_filter, image_filters.lbp_filter])
res['lbp_filter'].shape


(450, 600, 3)

In [5]:
filter_service.save_applied_filters(res, image_id="ISIC_0024306")

#### Apply filters to multiple images and save them

In [6]:
dataset_manager.load_cleaned_metadata(
    filter_melanoma_only=True,
)
dataset_manager.df['dx'].value_counts()

dx
nv     6703
mel    1113
Name: count, dtype: int64

In [7]:
nv_samples = dataset_manager.get_n_samples(6703, lesion_type='nv')
mel_samples = dataset_manager.get_n_samples(1113, lesion_type='mel')

sample_df = pd.concat([nv_samples, mel_samples]) # concat
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

sample_df['dx'].value_counts()

dx
nv     6703
mel    1113
Name: count, dtype: int64

In [ ]:
result_map = filter_service.filtered_image_generator(
    list_image_ids=sample_df['image_id'].tolist(),
    filter_list=[image_filters.glcm_filter, image_filters.gabor_filter, image_filters.lbp_filter],
    batch_size=10,
    asnpz=False
)